# Harness e Engenharia de Laço

O agente desta aula recebe uma tarefa em uma frase, planeja, escreve arquivos, roda comandos, escreve os próprios testes e se corrige até eles passarem.

O **harness** é o código em volta do modelo. Ele define quais ferramentas existem, quanto o agente pode gastar, quanto ele enxerga de cada resultado e quando a execução termina. Nada disso é inteligente, e é daí que vem a diferença entre uma demonstração e um agente que funciona duas vezes seguidas.

In [ ]:
# No Google Colab, descomente e rode uma vez.
!pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai" fastapi httpx

import os
import shutil
import subprocess
from pathlib import Path

from pydantic import BaseModel, Field

from agentkit.model import LLMAPI
from agentkit.agent import Agent
from agentkit.tools import tool

O agente escreve código, escolhe ferramentas e lê a saída do `pytest` para se corrigir. Modelos pequenos ignoram a instrução de escrever testes e encerram o turno sem ter agido.

O `max_tokens` é generoso porque o arquivo inteiro viaja dentro do argumento da chamada de ferramenta. Um teto apertado corta o argumento no meio e o JSON da chamada chega quebrado.

In [2]:
llm = LLMAPI("gpt-4.1-mini", temperature=0.0, max_tokens=3000)

# llm = LLMAPI(
#     "openai/gpt-oss-120b",
#     api_key=os.environ["GROQ_API_KEY"],
#     base_url="https://api.groq.com/openai/v1",
#     temperature=0.0,
#     max_tokens=3000,
# )

print(llm.model)

gpt-4.1-mini


A pasta do projeto é apagada no começo para o notebook rodar igual a partir de um kernel limpo.

In [3]:
WORKSPACE = Path("workspace/projeto")
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)

## Fundamentos

O agente se corrige sozinho enquanto trabalha. Ele escreve o código, roda os testes, lê a falha e reescreve, tudo dentro do próprio laço. O que ele decide também é quando aquilo acabou, e ele decide que acabou quando desistiu, quando explicou ao usuário como fazer o resto e quando acha que fez algo que não fez. Código quebrado entregue com confiança é o pior modo de falha.

Numa tarefa verificável existe um programa que dá a palavra final, e ele entra no laço:

```python
while not testes_passam():
    agent.run(messages)
```

O resto do notebook é o que cabe dentro dessas duas linhas.

## Ambiente e Insumos

`read_file` devolve o conteúdo de um arquivo do projeto. `write_file` substitui o arquivo inteiro. `run_shell` roda um comando na pasta e devolve o que saiu na tela, o que cobre listar com `ls`, procurar com `grep` e testar com `pytest`.

O tempo limite do `run_shell` existe porque o modelo às vezes roda algo que não termina. O corte da saída existe porque ela ocupa a janela de contexto na chamada seguinte.

In [4]:
@tool
def read_file(path: str) -> str:
    """Lê um arquivo do projeto e devolve o conteúdo."""
    file = WORKSPACE / path
    return file.read_text(encoding="utf-8") if file.exists() else f"{path} não existe"


@tool
def write_file(path: str, content: str) -> str:
    """Escreve um arquivo do projeto, substituindo o conteúdo anterior."""
    file = WORKSPACE / path
    file.parent.mkdir(parents=True, exist_ok=True)
    file.write_text(content, encoding="utf-8")
    return f"{path} gravado"


@tool
def run_shell(command: str) -> str:
    """Roda um comando no diretório do projeto e devolve a saída."""
    # O timeout existe porque o modelo às vezes roda algo que não termina.
    try:
        result = subprocess.run(
            command, shell=True, cwd=WORKSPACE, capture_output=True, text=True, timeout=20
        )
    except subprocess.TimeoutExpired:
        return "o comando passou de 20 segundos e foi interrompido"
    # Saída longa ocupa a janela de contexto sem informar.
    return (result.stdout + result.stderr).strip()[:800] or "ok"


TOOLS = [read_file, write_file, run_shell]

Exercitar as três com entrada conhecida evita descobrir um erro de caminho seis chamadas adiante.

In [5]:
print(write_file("teste.txt", "linha um\n"))
print(read_file("teste.txt"))
print(read_file("inexistente.py"))
print(run_shell("ls"))
print(run_shell("rm teste.txt"))

teste.txt gravado
linha um

inexistente.py não existe
teste.txt
ok


## Construção

São dois laços encaixados, e confundi-los é o erro mais comum ao ler código de agente.

```python
plan = create_plan(task)       # uma chamada, saída estruturada
while not testes_passam():     # laço do agente
    agent.run(messages)        # laço interno: ferramenta, observação, ferramenta
```

O laço interno vem pronto no `Agent` e é limitado por `max_steps`. Ele termina quando o modelo responde em texto. O laço de fora trata essa resposta como proposta e roda o `pytest` antes de aceitá-la.

### O plano

O plano é escrito antes de qualquer execução, e o esquema impõe a estrutura.

A terceira linha do `PLANNER_SYSTEM` é o que carrega a aula. Nenhum passo de teste está codificado no laço. Eles entram no plano porque o planejador sabe que o agente comprova o que escreve.

In [6]:
class Plan(BaseModel):
    steps: list[str] = Field(min_length=2)


PLANNER_SYSTEM = """Você escreve planos para um agente que edita arquivos e roda comandos numa pasta de projeto.
Cada passo é uma instrução que o agente executa sozinha, do começo ao fim.
O agente comprova o que escreve com testes automatizados, então o plano prevê escrever os testes e rodá-los.
Ainda não escreva código."""


def create_plan(task: str) -> Plan:
    """Pede o plano ao modelo, com o papel no sistema e a tarefa no usuário."""
    return llm.generate_structured(
        [{"role": "system", "content": PLANNER_SYSTEM}, {"role": "user", "content": task}],
        Plan,
        max_tokens=700,
    )

A tarefa é pequena de propósito. O interesse está no plano que sai de um pedido que não menciona teste.

In [7]:
TASK = """Escreva fibonacci.py com uma função fib(n) que devolva o n-ésimo número de Fibonacci,
com fib(0) igual a 0 e fib(1) igual a 1."""

plan = create_plan(TASK)

for number, step in enumerate(plan.steps, start=1):
    print(f"{number}. {step}")

1. Criar o arquivo fibonacci.py com a função fib(n) que calcula o n-ésimo número de Fibonacci, definindo fib(0) = 0 e fib(1) = 1.
2. Criar um arquivo de testes test_fibonacci.py que importe a função fib e contenha casos de teste para verificar se fib(0) retorna 0, fib(1) retorna 1, e outros valores como fib(5) e fib(10) retornam os valores corretos.
3. Executar os testes automatizados para garantir que a função fib está correta.


Os últimos passos escrevem o arquivo de teste e o executam. Quantos passos ao todo muda a cada execução.

### O executor

O `agent` é criado uma vez. O `max_steps` vale para o laço interno, e precisa caber o ciclo de ler o arquivo, escrever, rodar os testes, escrever de novo e rodar de novo antes de responder.

In [8]:
EXECUTOR_SYSTEM = """Você trabalha numa pasta de projeto, escrevendo arquivos e rodando comandos, e segue o plano que receber.
Leia um arquivo antes de reescrevê-lo, porque write_file substitui o arquivo inteiro.
Escreva os testes em arquivos com nome test_*.py e rode-os com `python -m pytest -q`.
Nunca rode um comando que não termina, como um servidor em primeiro plano."""

agent = Agent(llm, TOOLS, max_steps=10)

### A verificação

O `pytest` é o crítico. O código de saída é zero ou diferente de zero, sem interpretação e sem custo de token. Do `CompletedProcess` que volta, o `returncode` diz se passou e o `stdout` é o texto que vai para o modelo quando falhou.

In [9]:
def run_tests() -> subprocess.CompletedProcess:
    """Roda a suíte de testes do projeto."""
    return subprocess.run(
        "python -m pytest -q", shell=True, cwd=WORKSPACE, capture_output=True, text=True
    )

### O laço

O plano entra como contexto na primeira mensagem e o agente escolhe o próprio ritmo dentro dele. Quando ele responde em texto, o `Agent.run` devolve a conversa e o `pytest` decide se aquilo era o fim.

Enquanto a suíte estiver vermelha, a saída do erro volta como uma mensagem nova e o agente trabalha de novo sobre a mesma conversa. O `max_attempts` conta quantas vezes ele pode dizer que terminou.

In [10]:
def run_agent(task: str, plan: Plan, max_attempts: int = 5) -> list[dict]:
    """Executa o plano e só termina quando os testes passam."""
    messages = [
        {"role": "system", "content": EXECUTOR_SYSTEM},
        {"role": "user", "content": task + "\n\nPlano:\n" + "\n".join(plan.steps)},
    ]
    for attempt in range(max_attempts):
        messages = agent.run(messages)
        result = run_tests()
        print(f"verificação {attempt + 1}: {result.stdout.strip().splitlines()[-1]}")
        if result.returncode == 0:
            return messages
        messages = messages + [
            {"role": "user", "content": "Os testes falharam.\n" + result.stdout[-1500:]}
        ]
    return messages

## Execução e Traço

O traço mostra quais ferramentas foram chamadas e o que cada uma devolveu.

In [11]:
messages = run_agent(TASK, plan)


def show_trace(messages: list[dict]) -> None:
    """Imprime as chamadas de ferramenta e as observações da conversa."""
    for message in messages:
        for call in message.get("tool_calls", []):
            print(f"{call['name']}({str(call['arguments'])[:70]})")
        if message["role"] == "tool":
            print(f"   -> {' '.join(message['content'].split())[:80]}")


show_trace(messages)

verificação 1: 4 passed in 0.01s
write_file({'path': 'fibonacci.py', 'content': 'def fib(n):\n    if n == 0:\n    )
   -> fibonacci.py gravado
write_file({'path': 'test_fibonacci.py', 'content': 'import pytest\nfrom fibonacc)
   -> test_fibonacci.py gravado
run_shell({'command': 'python -m pytest -q'})
   -> /home/silvan/miniconda3/envs/ai/bin/python: No module named pytest
run_shell({'command': 'pip install pytest'})
   -> Collecting pytest Using cached pytest-9.1.1-py3-none-any.whl.metadata (7.6 kB) C
run_shell({'command': 'python -m pytest -q'})
   -> .... [100%] 4 passed


Cada linha de verificação marca um momento em que o agente disse que terminou. O traço mostra que ele roda o `pytest` sozinho durante o trabalho, com o mesmo comando do harness, e ainda assim é a verificação de fora que encerra o laço.

## Uma API de Notas

Esta tarefa erra mais na interface do que na lógica. O enunciado fixa o caminho das rotas e o formato da resposta, porque um pedido vago autoriza o agente a escolher, e aí ele acerta a tarefa e erra a integração.

In [12]:
API_TASK = """Escreva api.py com uma API em FastAPI que guarda notas de alunos em memória.
POST /notas recebe um JSON com nome e nota e guarda a nota.
GET /notas devolve todas as notas.
GET /media devolve a média das notas guardadas, no formato {"media": numero}.
Os caminhos são exatamente /notas e /media, sem barra no final."""

messages = run_agent(API_TASK, create_plan(API_TASK))
show_trace(messages)

verificação 1: 7 passed, 1 warning in 0.14s
write_file({'path': 'api.py', 'content': 'from fastapi import FastAPI, HTTPExcept)
   -> api.py gravado
write_file({'path': 'test_api.py', 'content': 'from fastapi.testclient import Tes)
   -> test_api.py gravado
run_shell({'command': 'python -m pytest -q'})
   -> ==================================== ERRORS ====================================
run_shell({'command': 'pip install fastapi[all]'})
   -> Collecting fastapi[all] Downloading fastapi-0.141.1-py3-none-any.whl.metadata (2
run_shell({'command': 'python -m pytest -q'})
   -> ....... [100


O traço é mais longo que o da primeira tarefa. Quando o `pytest` acusa uma falha no meio do turno, repare em qual arquivo o agente mexe em seguida: reescrever o teste custa o mesmo que reescrever o código, e só um dos dois cumpre o enunciado.

A verificação do laço roda a suíte inteira, então uma correção na API que quebrasse o Fibonacci apareceria aqui e o agente não conseguiria terminar.

In [13]:
print(read_file("api.py")[:600])

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

app = FastAPI()

class Nota(BaseModel):
    nome: str
    nota: float

notas: List[Nota] = []

@app.post("/notas")
def adicionar_nota(nota: Nota):
    if nota.nota < 0 or nota.nota > 10:
        raise HTTPException(status_code=400, detail="Nota deve ser entre 0 e 10")
    notas.append(nota)
    return {"message": "Nota adicionada com sucesso"}

@app.get("/notas")
def listar_notas():
    return notas

@app.get("/media")
def media_notas():
    if not notas:
        return {"media": 0}
    media = s


### Custo

Plano, executor e correção multiplicam chamadas rápido, e todas ficaram em `llm.usage`.

In [14]:
tokens_in = sum(call["tokens_in"] for call in llm.usage)
tokens_out = sum(call["tokens_out"] for call in llm.usage)
segundos = sum(call["seconds"] for call in llm.usage)

print(f"{len(llm.usage)} chamadas")
print(f"{tokens_in} tokens de entrada e {tokens_out} de saída")
print(f"{segundos:.1f} segundos")

14 chamadas
12733 tokens de entrada e 1660 de saída
40.6 segundos


## Exercício

Implemente um agente que monte a escala de plantão de uma semana, com uma pessoa por dia, e grave o resultado em `escala.json` na pasta do projeto, no formato `{"seg": "Ana", "ter": "Bruno", ...}`.

Ninguém trabalha em dois dias seguidos, e a semana é circular, então domingo e segunda contam como seguidos. Ninguém trabalha num dia em que está indisponível. Todo mundo trabalha pelo menos uma vez.

Você escreve o enunciado que vai para o agente, a função `check_schedule`, que lê o arquivo e devolve a lista de regras violadas, e o laço, na forma do `run_agent`, com `check_schedule` no lugar do `run_tests`. Rode, mostre o traço e a lista de violações de cada tentativa.

In [ ]:
PEOPLE = ["Ana", "Bruno", "Carla", "Davi"]
DAYS = ["seg", "ter", "qua", "qui", "sex", "sab", "dom"]
UNAVAILABLE = {"Ana": ["sab", "dom"], "Bruno": ["seg", "ter"], "Carla": ["qua", "qui"], "Davi": ["sex"]}